In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import kurtosis, spearmanr
import itertools
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. PARAMETRIC CONFIGURATION
# ==============================================================================
# 1. Get the directory where the current Jupyter Notebook is located
# Path.cwd() gets the current working directory of the notebook
notebook_dir = Path.cwd() 

# 2. Navigate up to the parent directory (crypto-perpetual-arbitrage)
project_dir = notebook_dir.parent 

# 3. Navigate into the 'data' folder
DATA_DIR = project_dir / "data"

# 4. Point to the specific parquet file
MASTER_PARQUET = DATA_DIR / "AVAX_TAGGED_MASTER.parquet"

# (Optional) Verify the paths are correct
print(f"Data Directory: {DATA_DIR}")
print(f"Master Parquet: {MASTER_PARQUET}")

GLOBAL_START = pd.Timestamp('2025-03-01 00:00:00')
GLOBAL_END = pd.Timestamp('2025-12-31 23:59:00')

ROLLING_WINDOW_MINS = 1440  # 24-Hour Ex-Ante Lookback
MIN_VALID_MINS = 720        # Require at least 12h of clean data inside the 24h window
ADJ_SPREAD_PREFIX = 'spread_padj_idx_mid_avg'

# 1. Taker Fees from VIP Table (in bps)
TAKER_FEES = {
    'binf': 1.70, 'byde': 3.00, 'okex': 1.75, 
    'kusw': 2.50, 'gate': 2.00, 'hype': 1.44
}
ASSUMED_SLIPPAGE_BPS = 0.5 # Per leg

# ==============================================================================
# PART 1: MACRO STYLIZED FACTS (MAB & ROLLING RELIABILITY)
# ==============================================================================
def run_part1_stylized_facts(df, all_pairs):
    print("\n" + "="*125)
    print(" 📘 PART 1: MACRO STYLIZED FACTS & MINIMUM ARBITRAGE BANDS (MAB)")
    print(" ⚙️  APPROACH: STATIC EX-POST WITH DAILY ROLLING SAMPLING")
    print("="*125)
    print(" 📖 METRICS GLOSSARY:")
    print(" - StdDev(bps)    : The unconditional volatility of the spread.")
    print(" - MAB(bps)       : Minimum Arbitrage Band. (Round-Trip Taker Fees + Assumed 2.0 bps Slippage).")
    print(" - Req. Sigma     : (MAB / StdDev). The theoretical Z-score magnitude required just to break even.")
    print(" - Static ADF     : Full-year Dickey-Fuller test (proves global cointegration).")
    print(" - Roll Stat %    : Percentage of rolling 24h windows that passed the ADF test (proves continuous tradability).")
    print("="*125)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        
        y_continuous = df[pair_col].where(active_mask)
        x_continuous = y_continuous.shift(1)
        acf1 = y_continuous.corr(x_continuous)
        
        clean_series = y_continuous.dropna()
        if len(clean_series) < 10000: continue
        
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"
        
        # 1. Static Global ADF
        adf_series = clean_series.iloc[::10] # Sampled for speed
        _, p_value_static, _, _, _, _ = adfuller(adf_series, maxlag=5)
        
        # 2. Rolling Daily ADF Reliability
        stationary_days, total_tested_days = 0, 0
        sample_points = range(1440, len(clean_series), 1440)
        for pt in sample_points:
            window = clean_series.iloc[pt-1440 : pt]
            if len(window) > 1000:
                try:
                    _, pval_roll, _, _, _, _ = adfuller(window, maxlag=1)
                    if pval_roll < 0.05: stationary_days += 1
                    total_tested_days += 1
                except: continue
        rolling_stat_pct = (stationary_days / total_tested_days * 100) if total_tested_days > 0 else 0
        
        # 3. MAB & Required Sigma
        fee_A = TAKER_FEES.get(exch_A, 2.0) 
        fee_B = TAKER_FEES.get(exch_B, 2.0)
        total_mab = ((fee_A + fee_B) * 2) + (ASSUMED_SLIPPAGE_BPS * 4)
        std_dev = clean_series.std()
        req_sigma = total_mab / std_dev if std_dev > 0 else np.nan

        results.append({
            'Pair': pair_name, 
            'Mean': clean_series.mean(), 
            'StdDev': std_dev,
            'MAB(bps)': total_mab,
            'Req. Sigma': req_sigma,
            'ACF(1)': acf1, 
            'Static ADF': p_value_static,
            'Roll Stat %': rolling_stat_pct, 
            'Active Days': len(clean_series) / 1440.0
        })

    res_df = pd.DataFrame(results).sort_values('StdDev')
    fmt = {'Mean': "{:.3f}", 'StdDev': "{:.3f}", 'MAB(bps)': "{:.2f}", 'Req. Sigma': "{:.2f}",
           'ACF(1)': "{:.4f}", 'Static ADF': "{:.4f}", 'Roll Stat %': "{:.1f}%", 'Active Days': "{:.1f}"}
    for col, f in fmt.items(): 
        if col in res_df.columns: res_df[col] = res_df[col].apply(lambda x: f.format(x) if pd.notna(x) else "N/A")
    print(res_df.to_string(index=False))

# ==============================================================================
# PART 2: TACTICAL SIGNAL FREQUENCY 
# ==============================================================================
def run_part2_signal_frequency(df, all_pairs):
    print("\n" + "="*125)
    print(" 📘 PART 2: TACTICAL SIGNAL FREQUENCY")
    print(" ⚙️  APPROACH: DYNAMIC ROLLING EX-ANTE (1440m WINDOW)")
    print("="*125)
    print(" 📖 METRICS GLOSSARY:")
    print(" - Naive Z > 3σ   : Breaches of a simple rolling standard deviation.")
    print(" - OU S-Score     : Breaches of the AR(1) filtered Ornstein-Uhlenbeck stochastic equilibrium.")
    print(" - Filter Toxicity: The percentage of Naive signals that the OU model correctly identifies as macro-drift (false positives).")
    print("="*125)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"

        y = df[pair_col].where(active_mask)
        x = y.shift(1)
        
        roll_mean = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()
        roll_std = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).std()
        roll_cov = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).cov(x)
        roll_var_x = x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        roll_var_y = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        
        rho_1 = (roll_cov / roll_var_x).clip(lower=1e-5, upper=0.999)
        m = (roll_mean - rho_1 * x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()) / (1 - rho_1)
        var_e = (roll_var_y - (rho_1**2) * roll_var_x).clip(lower=1e-8)
        sigma_eq = np.sqrt(var_e / (1 - rho_1**2))
        
        z_scores = ((y - roll_mean) / roll_std).dropna()
        s_scores = ((y - m) / sigma_eq).dropna()
        
        if len(s_scores) < 10000: continue
        z_3s, s_3s = (np.abs(z_scores) > 3.0).sum(), (np.abs(s_scores) > 3.0).sum()
        
        results.append({
            'Pair': pair_name, 'Naive Z > 3σ': z_3s, 'OU S-Score > 3σ': s_3s,
            'Naive Z > 4σ': (np.abs(z_scores) > 4.0).sum(), 'OU S-Score > 4σ': (np.abs(s_scores) > 4.0).sum(),
            'Filtered Toxicity %': (1 - (s_3s / max(z_3s, 1))) * 100
        })

    res_df = pd.DataFrame(results).sort_values('OU S-Score > 3σ', ascending=False)
    res_df['Filtered Toxicity %'] = res_df['Filtered Toxicity %'].apply(lambda x: f"{x:.1f}%")
    print(res_df.to_string(index=False))

# ==============================================================================
# PART 3: OU MODEL VALIDATION
# ==============================================================================
def run_part3_ou_validation(df, all_pairs):
    print("\n" + "="*125)
    print(" 📘 PART 3: OU STOCHASTIC MODEL VALIDATION & RESIDUAL DIAGNOSTICS")
    print(" ⚙️  APPROACH: STATIC EX-POST (FULL SAMPLE REGRESSION)")
    print("="*125)
    print(" 📖 METRICS GLOSSARY:")
    print(" - AR(1) Beta   : The mean-reversion speed. Closer to 1.0 = slower decay.")
    print(" - Mean Delta   : Absolute difference between Theoretical limits and Empirical physical order book centers.")
    print("                  Values near 0 prove the model is perfectly anchored to reality.")
    print("="*125)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"

        y = df[pair_col].where(active_mask)
        x = y.shift(1)
        df_reg = pd.DataFrame({'y': y, 'x': x}).dropna()
        if len(df_reg) < 10000: continue

        model = sm.OLS(df_reg['y'], sm.add_constant(df_reg['x'])).fit()
        alpha, beta = model.params.iloc[0], model.params.iloc[1]
        
        if beta >= 1 or beta <= 0: continue
        theo_mean = alpha / (1 - beta)
        emp_mean = df_reg['y'].mean()
        lb_pval = acorr_ljungbox(model.resid, lags=[10], return_df=True)['lb_pvalue'].iloc[0]
        
        results.append({
            'Pair': pair_name, 'AR(1) Beta': beta, 'Theo Mean(μ)': theo_mean,
            'Emp Mean(x̄)': emp_mean, 'Mean Delta(Δ)': abs(theo_mean - emp_mean), 'Ljung-Box': lb_pval
        })

    res_df = pd.DataFrame(results).sort_values('AR(1) Beta')
    fmt = {'AR(1) Beta': "{:.4f}", 'Theo Mean(μ)': "{:.3f}", 'Emp Mean(x̄)': "{:.3f}", 
           'Mean Delta(Δ)': "{:.4f}", 'Ljung-Box': "{:.4f}"}
    for col, f in fmt.items(): res_df[col] = res_df[col].apply(f.format)
    print(res_df.to_string(index=False))

# ==============================================================================
# PART 4: MODEL USEFULNESS & RANK CORRELATION
# ==============================================================================
def calculate_anchored_empirical_ttr(y_series, m_series, s_scores, sigma_thresh):
    triggers = np.where(np.abs(s_scores) >= sigma_thresh)[0]
    clean_triggers = []
    last_t = -60
    for t in triggers:
        if t - last_t >= 60:
            clean_triggers.append(t)
            last_t = t
            
    if not clean_triggers: return np.nan
    ttrs = []
    y_vals, m_vals, s_vals = y_series.values, m_series.values, s_scores.values
    max_len = len(y_vals)
    
    for t in clean_triggers:
        anchor_m = m_vals[t]
        direction = -1 if s_vals[t] > 0 else 1
        max_search = min(t + 2880, max_len)
        forward_y = y_vals[t+1 : max_search]
        
        if direction == -1: crossings = np.where(forward_y < anchor_m)[0]
        else: crossings = np.where(forward_y > anchor_m)[0]
            
        if len(crossings) > 0: ttrs.append(crossings[0] + 1)
            
    return np.median(ttrs) if ttrs else np.nan

def run_part4_theoretical_vs_empirical(df, all_pairs):
    print("\n" + "="*125)
    print(" 📘 PART 4: MODEL USEFULNESS & RANK CORRELATION (THEORETICAL VS EMPIRICAL TTR)")
    print("="*125)
    print(" 📖 METRICS GLOSSARY:")
    print(" - Theo Half-Life : Mathematical half-life of the AR(1) decay.")
    print(" - Anchored TTR   : Realized physical time-to-reversion once a sigma threshold is breached.")
    print(" - Spearman Corr  : Proves if pairs with mathematically fast decay actually decay faster in physical reality.")
    print("="*125)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"
        
        y = df[pair_col].where(active_mask)
        x = y.shift(1)
        
        roll_mean = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()
        roll_cov = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).cov(x)
        roll_var_x = x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        roll_var_y = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        
        rho_1_series = (roll_cov / roll_var_x).clip(lower=1e-5, upper=0.999)
        avg_rho_1 = rho_1_series.mean()
        kappa = -np.log(avg_rho_1) if avg_rho_1 > 0 and pd.notna(avg_rho_1) else np.nan
        theo_hl = np.log(2) / kappa if pd.notna(kappa) and kappa > 0 else np.nan
        
        m = (roll_mean - rho_1_series * x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()) / (1 - rho_1_series)
        var_e = (roll_var_y - (rho_1_series**2) * roll_var_x).clip(lower=1e-8)
        sigma_eq = np.sqrt(var_e / (1 - rho_1_series**2))
        s_scores = ((y - m) / sigma_eq)
        
        if len(s_scores.dropna()) < 10000: continue
        
        ttr_1s = calculate_anchored_empirical_ttr(y, m, s_scores, 1.0)
        ttr_2s = calculate_anchored_empirical_ttr(y, m, s_scores, 2.0)
        ttr_3s = calculate_anchored_empirical_ttr(y, m, s_scores, 3.0)
        
        results.append({
            'Pair': pair_name, 'Theo Half-Life': theo_hl,
            'TTR 1σ': ttr_1s, 'TTR 2σ': ttr_2s, 'TTR 3σ': ttr_3s
        })

    if not results: return print("[!] No pairs met the minimum sample size constraint.")

    res_df = pd.DataFrame(results).sort_values('Theo Half-Life')
    corr_1s = spearmanr(res_df['Theo Half-Life'], res_df['TTR 1σ'], nan_policy='omit')[0]
    corr_2s = spearmanr(res_df['Theo Half-Life'], res_df['TTR 2σ'], nan_policy='omit')[0]
    corr_3s = spearmanr(res_df['Theo Half-Life'], res_df['TTR 3σ'], nan_policy='omit')[0]
    
    fmt = lambda x: f"{x:.1f}" if pd.notna(x) else "N/A"
    for c in ['Theo Half-Life', 'TTR 1σ', 'TTR 2σ', 'TTR 3σ']: res_df[c] = res_df[c].apply(fmt)
            
    print(res_df.to_string(index=False))
    print("="*125)
    print(f" 📈 SPEARMAN RANK CORRELATION (Theo HL vs 1σ Bulk Regime TTR): {corr_1s:.3f}")
    print(f" 📈 SPEARMAN RANK CORRELATION (Theo HL vs 2σ Void Regime TTR): {corr_2s:.3f}")
    print(f" 📈 SPEARMAN RANK CORRELATION (Theo HL vs 3σ Tail Regime TTR): {corr_3s:.3f}")
    print("="*125)

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    print("🚀 Ingesting Master Dataset...")
    df = pd.read_parquet(MASTER_PARQUET).loc[GLOBAL_START:GLOBAL_END]
    exchanges = [c.split('_')[-1] for c in df.columns if c.startswith('mid_')]
    all_pairs = list(itertools.combinations(exchanges, 2))
    
    run_part1_stylized_facts(df, all_pairs)
    run_part2_signal_frequency(df, all_pairs)
    run_part3_ou_validation(df, all_pairs)
    run_part4_theoretical_vs_empirical(df, all_pairs)

Data Directory: C:\Users\tz_ho\crypto-perpetual-arbitrage\data
Master Parquet: C:\Users\tz_ho\crypto-perpetual-arbitrage\data\AVAX_TAGGED_MASTER.parquet
🚀 Ingesting Master Dataset...

 📘 PART 1: MACRO STYLIZED FACTS & MINIMUM ARBITRAGE BANDS (MAB)
 ⚙️  APPROACH: STATIC EX-POST WITH DAILY ROLLING SAMPLING
 📖 METRICS GLOSSARY:
 - StdDev(bps)    : The unconditional volatility of the spread.
 - MAB(bps)       : Minimum Arbitrage Band. (Round-Trip Taker Fees + Assumed 2.0 bps Slippage).
 - Req. Sigma     : (MAB / StdDev). The theoretical Z-score magnitude required just to break even.
 - Static ADF     : Full-year Dickey-Fuller test (proves global cointegration).
 - Roll Stat %    : Percentage of rolling 24h windows that passed the ADF test (proves continuous tradability).
     Pair    Mean StdDev MAB(bps) Req. Sigma ACF(1) Static ADF Roll Stat % Active Days
BINA/BYBI   0.144  2.460    10.00       4.06 0.9092     0.0000      100.0%       137.4
BYBI/GATE   2.803  2.613    10.00       3.83 0.9